In [1]:
from exoatlas import * 

In [2]:
e = TransitingExoplanets()['TOI3884b']

Loaded standardized table from /Users/zabe0091/exoatlas-data/data/standardized-Exoplanets.txt


In [3]:
e

✨ TOI3884b | 1 elements ✨

In [4]:
filters = {
'g':[400, 550]*u.nm,
'r':[550, 700]*u.nm,
'i':[700, 820]*u.nm,
'Zs':[820, 920]*u.nm
}

In [5]:
from rainbowconnection import Star

In [6]:
def crudely_estimate_magnitude(teff=5800*u.K, radius=1*u.Rsun, distance=10*u.pc, wavelength = [.4, .9]*u.micron, N=10):

    fine_wavelength_grid = np.linspace(wavelength[0], wavelength[1], N)
    photon_energy = con.h*con.c/fine_wavelength_grid/u.ph

    s = Star(teff=teff, radius=radius).at(distance)
    flux = s.spectrum(wavelength=fine_wavelength_grid)
    flux_photons = (flux/photon_energy).decompose()

    f_nu = 3631*u.Jy
    f_lambda = con.c/fine_wavelength_grid**2*f_nu
    flux_zeropoint_photons = (f_lambda/photon_energy).decompose()

    integrated_flux_zeropoint_photons = np.trapezoid(flux_zeropoint_photons, fine_wavelength_grid).decompose()
    integrated_flux_photons = np.trapezoid(flux_photons, fine_wavelength_grid).decompose()

    magnitude = -2.5*np.log10(integrated_flux_photons/integrated_flux_zeropoint_photons)

    return magnitude

In [7]:
for f, w in filters.items():
    print(f, crudely_estimate_magnitude(teff=e.stellar_teff(), radius=e.stellar_radius(), distance=e.distance(), wavelength=w))

g 16.234618956107255
r 14.78108976843237
i 13.412874715192697
Zs 12.839768933333367


In [8]:
!open .